In [ ]:
from features import build_features, feature_groups

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_DIR = "../data"

In [ ]:
train_raw = pd.read_excel(f"{DATA_DIR}/Assignment_Train.xlsx", na_values=["NaT"])
test_raw  = pd.read_excel(f"{DATA_DIR}/Assignment_Test.xlsx",  na_values=["NaT"])

assert train_raw.shape == (10_000, 94)
assert test_raw.shape == (5_000, 93)
assert list(test_raw.columns) == [c for c in train_raw.columns if c != "Target"]
assert train_raw["UID"].is_unique and test_raw["UID"].is_unique
print("train:", train_raw.shape, "| test:", test_raw.shape)

In [ ]:
has_label = train_raw["Target"].notna()
train = train_raw.loc[has_label].copy()
train["Target"] = train["Target"].astype("int8")

assert len(train) == 9_992
assert set(train["Target"].unique()) == {0, 1}
print("Rows dropped:", int((~has_label).sum()), "| positives:", int(train["Target"].sum()),
      "| positive rate:", round(train["Target"].mean(), 4))

In [ ]:
dev, holdout = train_test_split(
    train, test_size=0.30, stratify=train["Target"], random_state=RANDOM_STATE
)

assert len(dev) == 6_994 and len(holdout) == 2_998
assert set(dev["UID"]).isdisjoint(holdout["UID"])
print("dev:", dev.shape, round(dev["Target"].mean(), 4), "| holdout:", holdout.shape, round(holdout["Target"].mean(), 4))

Baseline Results

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc"}            

def make_logreg(sentinel_mode="flag"):
    """Logistic regression with preprocessing learned inside each CV training fold."""
    numeric, flags, categorical = feature_groups(sentinel_mode)
    preprocess = ColumnTransformer([
        ("num",  Pipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale",  StandardScaler())]), numeric),
        ("flag", SimpleImputer(strategy="most_frequent"), flags),
        ("cat",  OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    return Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

def evaluate(model, X, y):
    """Mean and std of each metric across the 5 CV folds."""
    res = cross_validate(model, X, y, cv=CV, scoring=SCORING)
    return {f"{m}_mean": res[f"test_{m}"].mean() for m in SCORING} | \
           {f"{m}_std":  res[f"test_{m}"].std()  for m in SCORING}

In [ ]:
y_dev  = dev["Target"]
X_flag = build_features(dev, "flag").drop(columns=["UID", "Target"])
X_raw  = build_features(dev, "raw").drop(columns=["UID", "Target"])

results = pd.DataFrame({
    "dummy (base rate)":     evaluate(DummyClassifier(strategy="prior"), X_flag, y_dev),
    "logreg, -1 raw (a)":    evaluate(make_logreg("raw"),  X_raw,  y_dev),
    "logreg, -1 flag (c)":   evaluate(make_logreg("flag"), X_flag, y_dev),
}).T.round(3)
results

Raw + selected 22 features

In [ ]:
from features import (
    build_features,
    feature_groups,
    handle_sentinels,
    add_platform_share,
    encode_ordinals,
    clean_categories,
    MODEL_FEATURES
)

FIXED_EXCLUDE = [
    "UID", "Target",
    "name", "Phone to Name.UPIname", "Email Name Attributes.phoneNumber",          # PII
    "Phone Network.currentNetworkRegion", "Phone Network.portedFromNetworkRegion",  # region (D34)
    "Phone Name Attributes.revocationMonth",                                        # a date, used as evidence only
]
# Columns with at most one known value; decided from dev only, then fixed
CONSTANT_COLS = [c for c in dev.columns if dev[c].nunique() <= 1]

def build_all_columns(data):
    """Option (ii): our engineered features plus every other usable column."""
    out = handle_sentinels(data, mode="flag")
    out = add_platform_share(out)
    out = encode_ordinals(out)
    out = clean_categories(out)
    drop = [c for c in FIXED_EXCLUDE + CONSTANT_COLS if c in out.columns]
    return out.drop(columns=drop)

X_all = build_all_columns(dev)
CAT_ALL = X_all.select_dtypes(include=["str", "object"]).columns.tolist()
NUM_ALL = [c for c in X_all.columns if c not in CAT_ALL]

assert set(MODEL_FEATURES) <= set(X_all.columns)                          # the 22 are a subset (option ii)
assert list(build_all_columns(test_raw).columns) == list(X_all.columns)   # test gets the same columns
print("constant columns dropped:", CONSTANT_COLS)
print(X_all.shape, "| numeric:", len(NUM_ALL), "| categorical:", len(CAT_ALL))


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from catboost import CatBoostClassifier

def fill_missing_text(X, cats):
    """Missing text -> 'missing' (row-wise, nothing learned, so leakage-safe)."""
    X = X.copy()
    X[cats] = X[cats].astype("object").where(X[cats].notna(), "missing")
    return X

def make_model(kind, numeric, categorical):
    """One untuned model of the given kind, with preprocessing suited to it."""
    if kind == "catboost":                                   # handles categories and NaN itself
        return CatBoostClassifier(iterations=500, random_seed=RANDOM_STATE, verbose=0,
                                  cat_features=categorical, allow_writing_files=False)
    num_steps = [] if kind == "hgb" else [("impute", SimpleImputer(strategy="median"))]   # HGB handles NaN natively
    if kind == "logreg":
        num_steps.append(("scale", StandardScaler()))        # only the linear model needs scaling
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps) if num_steps else "passthrough", numeric),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20), categorical),
    ])
    model = {
        "logreg": LogisticRegression(max_iter=2000),
        "rf":     RandomForestClassifier(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
        "hgb":    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }[kind]
    return Pipeline([("prep", prep), ("model", model)])

def cv_evaluate(kind, X, y, numeric, categorical):
    """Manual 5-fold CV (same folds as before). Returns metrics and out-of-fold predictions."""
    oof = pd.Series(np.nan, index=X.index)
    pr, roc = [], []
    for train_idx, val_idx in CV.split(X, y):
        model = make_model(kind, numeric, categorical)       # a fresh, untrained model every fold
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        p = model.predict_proba(X.iloc[val_idx])[:, 1]
        oof.iloc[val_idx] = p
        pr.append(average_precision_score(y.iloc[val_idx], p))
        roc.append(roc_auc_score(y.iloc[val_idx], p))
    return {"pr_auc_mean": np.mean(pr), "pr_auc_std": np.std(pr),
            "roc_auc_mean": np.mean(roc), "roc_auc_std": np.std(roc)}, oof

# The two feature sets
num_sel, flag_sel, cat_sel = feature_groups("flag")
FEATURE_SETS = {
    "selected 22": (fill_missing_text(X_flag, cat_sel), num_sel + flag_sel, cat_sel),
    "all 84":      (fill_missing_text(X_all, CAT_ALL),  NUM_ALL,            CAT_ALL),
}

grid_rows, oof_preds = [], {}
for kind in ["logreg", "rf", "hgb", "catboost"]:
    for set_name, (X, numeric, categorical) in FEATURE_SETS.items():
        metrics, oof = cv_evaluate(kind, X, y_dev, numeric, categorical)
        grid_rows.append({"model": kind, "features": set_name, **metrics})
        oof_preds[(kind, set_name)] = oof

grid = pd.DataFrame(grid_rows).round(3)
grid.pivot(index="model", columns="features", values="pr_auc_mean")

Every model score lies between 0.199 and 0.235 at a spread of 0.036 which is approx. equal to fold-to-fold noise (~0.03).
Since most complex models are tied on the scores and logreg out-performs them, it can be said that the relationship between data and the target is quite linear and the simple model captures them better.
Adding the other raw features to the selected 22 adds noise in the data which detoriates the performnace of the models.

In [ ]:
grid.sort_values("pr_auc_mean", ascending=False)

Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

X_all_filled = fill_missing_text(X_all, CAT_ALL)
fold_drops = []

for train_idx, val_idx in CV.split(X_all_filled, y_dev):
    model = make_model("rf", NUM_ALL, CAT_ALL).fit(X_all_filled.iloc[train_idx], y_dev.iloc[train_idx])
    result = permutation_importance(
        model, X_all_filled.iloc[val_idx], y_dev.iloc[val_idx],
        scoring="average_precision",   # importance = how much PR-AUC drops when the column is shuffled
        n_repeats=5,                   # shuffle each column 5 times and average (shuffles are random)
        random_state=RANDOM_STATE,
    )
    fold_drops.append(pd.Series(result.importances_mean, index=X_all_filled.columns))

drops = pd.concat(fold_drops, axis=1)
importance = pd.DataFrame({
    "pr_auc_drop":     drops.mean(axis=1),          # average over the 5 folds
    "std_over_folds":  drops.std(axis=1),
    "folds_positive":  (drops > 0).sum(axis=1),     # in how many of 5 folds shuffling hurt
    "in_selected_22":  drops.index.isin(MODEL_FEATURES),
}).sort_values("pr_auc_drop", ascending=False).round(4)

importance.head(15)

In [ ]:
CANDIDATES = [
    "Phone Social Advance.instagram", "Email Social Advance.amazon", "email_platform_known",
    "Mobile GIN.l2Confidence", "Mobile GIN.l1Confidence", "Phone Social Advance.housing",
]
X_plus = fill_missing_text(X_all[MODEL_FEATURES + CANDIDATES], cat_sel)
metrics_plus, _ = cv_evaluate("logreg", X_plus, y_dev, num_sel + flag_sel + CANDIDATES, cat_sel)
pd.DataFrame({"selected 22": grid.query("model == 'logreg' and features == 'selected 22'").iloc[0][2:],
              "22 + 6 candidates": pd.Series(metrics_plus)}).T.round(3)

Permutation importance on an 84-column Random Forest confirmed that the six most important features were already in the selected set. Six smaller candidates were tested together. They added +0.005 PR-AUC, well within the ±0.03 fold variation, so keeping the simpler 22-feature model.

In [ ]:
def cv_evaluate(kind, X, y, numeric, categorical, params=None):           # new: params
    """Manual 5-fold CV (same folds as before). Returns metrics and out-of-fold predictions."""
    oof = pd.Series(np.nan, index=X.index)
    pr, roc = [], []
    for train_idx, val_idx in CV.split(X, y):
        model = make_model(kind, numeric, categorical)
        if params:
            model.set_params(**params)                                    # new: e.g. class_weight
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        p = model.predict_proba(X.iloc[val_idx])[:, 1]
        oof.iloc[val_idx] = p
        pr.append(average_precision_score(y.iloc[val_idx], p))
        roc.append(roc_auc_score(y.iloc[val_idx], p))
    return {"pr_auc_mean": np.mean(pr), "pr_auc_std": np.std(pr),
            "roc_auc_mean": np.mean(roc), "roc_auc_std": np.std(roc)}, oof

X_sel_filled = fill_missing_text(X_flag, cat_sel)
cols = (X_sel_filled, y_dev, num_sel + flag_sel, cat_sel)

m_none, oof_lr          = cv_evaluate("logreg", *cols)
m_balanced, oof_lr_bal  = cv_evaluate("logreg", *cols, params={"model__class_weight": "balanced"})

pd.DataFrame({"no weighting": m_none, "class_weight='balanced'": m_balanced}).T.round(3)

Class weighting didn't change ranking performance (PR-AUC 0.234 vs 0.235), as expected, since it mainly rescales probabilities. Imbalance is addressed through metric choice (PR-AUC) and threshold selection.

In [ ]:
WHATSAPP = "Phone Social Advance.whatsapp"
BILLING  = "Phone Network.numberBillingType"
REGION   = "Phone Network.currentNetworkRegion"

def run_variant(drop=(), add_region=False):
    """Logistic regression on the 22 features, minus `drop`, optionally plus raw region."""
    numeric     = [c for c in num_sel + flag_sel if c not in drop]
    categorical = [c for c in cat_sel if c not in drop]
    X = X_flag[numeric + categorical].copy()
    if add_region:
        X[REGION] = dev[REGION]                  # same rows: X_flag keeps dev's index
        categorical = categorical + [REGION]
    metrics, _ = cv_evaluate("logreg", fill_missing_text(X, categorical), y_dev, numeric, categorical)
    return metrics

variants = {
    "baseline (22 features)":   run_variant(),
    "without WhatsApp":         run_variant(drop=[WHATSAPP]),
    "without billing":          run_variant(drop=[BILLING]),
    "without both":             run_variant(drop=[WHATSAPP, BILLING]),
    "with region (D34)":        run_variant(add_region=True),
}
sensitivity = pd.DataFrame(variants).T.round(3)
sensitivity["pr_auc_change"] = (sensitivity["pr_auc_mean"] - sensitivity.loc["baseline (22 features)", "pr_auc_mean"]).round(3)
sensitivity

Region was excluded on fairness grounds: it can act as a proxy for protected characteristics and is hard to justify as a basis for an individual credit decision. A sensitivity check confirmed the exclusion has no measurable cost (PR-AUC +0.002, within fold noise; ROC-AUC slightly lower).

Ablation study

In [ ]:
from features import NOT_FOUND_FLAGS

SHARES    = ["email_platform_share", "phone_platform_share"]
NOT_FOUND = list(NOT_FOUND_FLAGS)
ORDINALS  = ["Email Attributes.footprint", "Phone Name Attributes.phoneFootprintStrengthOverall",
             "Email Name Attributes.emailFootprintStrength", "footprint_no_records"]

ablation_variants = {
    "baseline (22 features)":      run_variant(),
    "without platform shares":     run_variant(drop=SHARES),
    "without not-found flags":     run_variant(drop=NOT_FOUND),
    "without footprint ordinals":  run_variant(drop=ORDINALS),
    "without all engineered":      run_variant(drop=SHARES + NOT_FOUND + ORDINALS),
}
ablation = pd.DataFrame(ablation_variants).T.round(3)
ablation["pr_auc_change"] = (ablation["pr_auc_mean"] - ablation.loc["baseline (22 features)", "pr_auc_mean"]).round(3)
ablation

Each engineered feature showed a clear univariate relationship with delinquency, but an ablation showed none added measurable CV performance once the raw features were present (all changes within ±0.004, fold std ≈ 0.033), most likely because they overlap with WhatsApp, uniqueHits and email age.

Ensemble approach

In [ ]:
from scipy.stats import spearmanr

lr_oof = oof_preds[("logreg", "selected 22")]
rf_oof = oof_preds[("rf", "selected 22")]

# 1. Do they rank people differently?
rho = spearmanr(lr_oof, rf_oof).statistic
print("Rank correlation, logistic regression vs Random Forest:", round(rho, 3))

# 2. Rank average: each model's predictions turned into percentiles (0-1), then averaged
ensemble_oof = (lr_oof.rank(pct=True) + rf_oof.rank(pct=True)) / 2

def fold_pr_auc(pred):
    """PR-AUC on each validation fold (same folds as everything else)."""
    scores = [average_precision_score(y_dev.iloc[val], pred.iloc[val]) for _, val in CV.split(X_flag, y_dev)]
    return round(np.mean(scores), 3), round(np.std(scores), 3)

pd.DataFrame({
    "logistic regression": fold_pr_auc(lr_oof),
    "random forest":       fold_pr_auc(rf_oof),
    "rank average":        fold_pr_auc(ensemble_oof),
}, index=["pr_auc_mean", "pr_auc_std"]).T

The rank average scored +0.006 PR-AUC on average, but it was better in only 3 of 5 folds, and the gain came almost entirely from one fold (+0.031; ≈ 0 in the others). The single logistic regression was kept.

Logistic regression and Random Forest ranked borrowers fairly differently (Spearman 0.74), so a rank-average ensemble was tested. It gave +0.006 PR-AUC, within fold noise, with higher variance, so the single logistic regression was kept.

In [ ]:
final_model = make_model("logreg", num_sel + flag_sel, cat_sel)
final_model.fit(X_sel_filled, y_dev)

names = final_model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(final_model.named_steps["model"].coef_[0], index=names)

coef_table = pd.DataFrame({
    "coef":       coefs,
    "odds_ratio": np.exp(coefs),     # multiplicative effect on the odds of delinquency
}).sort_values("coef", ascending=False).round(3)
coef_table

The strongest effects are absence of WhatsApp (≈ 4.8× the odds of delinquency), billing type (unknown ≈ 4.2× and postpaid ≈ 0.6× the odds of prepaid), being 'not found' in the vendor's email–name lookup, and a weaker online presence (fewer unique hits, younger email). Some individual coefficients have counter-intuitive signs, e.g. email platform share and first-name match. These are explained by strong correlations (ρ = 0.94 with uniqueHits; ρ = −0.84 with the not-found flag), so these features are interpreted as groups. All effects are associations, not causes.

In [ ]:
def cutoff_table(y, scores, shares=(0.05, 0.10, 0.15, 0.20, 0.30, 0.40)):
    """For each 'flag the riskiest X%' rule: how many delinquents are caught, and how many flags are wrong."""
    rows = []
    for share in shares:
        cutoff  = scores.quantile(1 - share)          # score above which the top `share` of people fall
        flagged = scores >= cutoff
        caught  = int((flagged & (y == 1)).sum())
        rows.append({
            "flag_top":        f"{share:.0%}",
            "people_flagged":  int(flagged.sum()),
            "delinquents_caught": caught,
            "recall":          caught / int((y == 1).sum()),     # share of ALL delinquents caught
            "precision":       caught / int(flagged.sum()),      # share of flagged who ARE delinquent
            "good_borrowers_flagged": int(flagged.sum()) - caught,  # false positives
        })
    return pd.DataFrame(rows).round(3)

cutoff_table(y_dev, oof_lr)

In [ ]:
# 1. Cut-offs fixed from dev BEFORE looking at the holdout (D71)
CUT_DECLINE = oof_lr.quantile(0.95)    # risk score of the top 5% boundary
CUT_REVIEW  = oof_lr.quantile(0.80)    # top 20% boundary
print("decline if p >=", round(CUT_DECLINE, 3), "| review if p >=", round(CUT_REVIEW, 3))

# 2. Final model: fit on all of dev, then score the holdout once
final_model = make_model("logreg", num_sel + flag_sel, cat_sel).fit(X_sel_filled, y_dev)
X_hold = fill_missing_text(build_features(holdout, "flag").drop(columns=["UID", "Target"]), cat_sel)
y_hold = holdout["Target"]
p_hold = pd.Series(final_model.predict_proba(X_hold)[:, 1], index=X_hold.index)

print("HOLDOUT  PR-AUC:", round(average_precision_score(y_hold, p_hold), 3),
      "| ROC-AUC:", round(roc_auc_score(y_hold, p_hold), 3))

# 3. Apply the tiers and count outcomes
tier = pd.Series(np.select([p_hold >= CUT_DECLINE, p_hold >= CUT_REVIEW], ["1 decline", "2 review"],
                           default="3 approve"), index=p_hold.index, name="tier")
outcome = pd.crosstab(tier, y_hold.rename("delinquent"), margins=True)
outcome["delinquency_rate"] = (outcome[1] / outcome["All"]).round(3)
outcome

In [ ]:
# --- Stage 7.1: final model on ALL labelled data (D74) ---
X_train = fill_missing_text(build_features(train, "flag").drop(columns=["UID", "Target"]), cat_sel)
y_train = train["Target"]

# Reference distribution: out-of-fold predictions on all 9,992 rows (scores the model gives to "unseen" people)
reference_oof = pd.Series(np.nan, index=X_train.index)
for train_idx, val_idx in CV.split(X_train, y_train):
    fold_model = make_model("logreg", num_sel + flag_sel, cat_sel).fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
    reference_oof.iloc[val_idx] = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
REFERENCE_SORTED = np.sort(reference_oof.values)

def risk_score(p):
    """1-100 score = percentile of predicted risk within the reference distribution (D75). 100 = highest risk."""
    share_at_or_below = np.searchsorted(REFERENCE_SORTED, p, side="right") / len(REFERENCE_SORTED)
    return np.clip(np.ceil(share_at_or_below * 100), 1, 100).astype(int)

final_model = make_model("logreg", num_sel + flag_sel, cat_sel).fit(X_train, y_train)

# --- Stage 7.2: score the test set ---
test_fe = build_features(test_raw, "flag")
X_test  = fill_missing_text(test_fe.drop(columns=["UID"]), cat_sel)
p_test  = final_model.predict_proba(X_test)[:, 1]
test_score = risk_score(p_test)

# --- Checks ---
assert len(test_score) == len(test_raw) == 5_000
assert (test_fe["UID"].values == test_raw["UID"].values).all()          # same rows, same order
assert test_score.min() >= 1 and test_score.max() <= 100
assert np.all(np.diff(risk_score(np.sort(p_test))) >= 0)                 # higher risk never gets a lower score
print("Reference share with score >= 96:", round((risk_score(reference_oof.values) >= 96).mean(), 3))   # expect 0.05

tiers = pd.cut(test_score, bins=[0, 80, 95, 100], labels=["approve (1-80)", "review (81-95)", "decline (96-100)"])
pd.Series(tiers).value_counts(normalize=True).round(3)

Training data is 94% Airtel customers, while the test set is 45% Airtel, 39% Jio and 16% other operators. Some features behave differently by operator; for example, billing type is never available for Vodafone, IDEA or BSNL. So test scores (7.6% in the decline band, against 5% in the reference) partly reflect operator mix, and the model's accuracy on non-Airtel applicants is unverified. Recommendation: validate per operator once outcomes for this population are available, and collect more labelled non-Airtel data.

Airtel vs Non-Airtel (train set (dev + holdout))

In [ ]:
network_group = train["Phone Network.currentNetworkName"].where(
    train["Phone Network.currentNetworkName"] == "Airtel", "non-Airtel")      # Jio, Vodafone, IDEA, BSNL, missing

rng = np.random.default_rng(RANDOM_STATE)
def bootstrap_roc_ci(y, p, n_boot=500):
    """95% interval for ROC-AUC by resampling people with replacement."""
    y, p = np.asarray(y), np.asarray(p)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() != y[idx].max():                 # need both classes in the resample
            scores.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(scores, [2.5, 97.5])

rows = []
for group in ["Airtel", "non-Airtel"]:
    mask = network_group == group
    y_g, p_g = y_train[mask], reference_oof[mask]
    low, high = bootstrap_roc_ci(y_g, p_g)
    rows.append({
        "group": group, "people": int(mask.sum()), "delinquents": int(y_g.sum()),
        "base_rate": y_g.mean(),
        "pr_auc": average_precision_score(y_g, p_g),
        "pr_auc_lift": average_precision_score(y_g, p_g) / y_g.mean(),   # PR-AUC relative to that group's random level
        "roc_auc": roc_auc_score(y_g, p_g),
        "roc_auc_95ci": f"{low:.2f} - {high:.2f}",
    })
pd.DataFrame(rows).round(3)


Does model need network feature

In [ ]:
network_check = pd.DataFrame({
    "baseline (22 features)": run_variant(),
    "without network":        run_variant(drop=["Phone Network.currentNetworkName"]),
}).T.round(3)
network_check["pr_auc_change"] = (network_check["pr_auc_mean"] - network_check.loc["baseline (22 features)", "pr_auc_mean"]).round(3)
network_check

holdout with 21 features re-test

In [ ]:
# Final feature set: the 22 minus network (D80)
cat_final = [c for c in cat_sel if c != "Phone Network.currentNetworkName"]
final_cols = (num_sel + flag_sel, cat_final)
print("final model features:", len(num_sel + flag_sel + cat_final))          # expect 21

# 1. Cut-offs fixed from dev out-of-fold scores of the 21-feature model, BEFORE looking at the holdout
metrics_21, oof_21 = cv_evaluate("logreg", X_sel_filled, y_dev, *final_cols)
CUT_DECLINE_21 = oof_21.quantile(0.95)
CUT_REVIEW_21  = oof_21.quantile(0.80)
print("dev CV PR-AUC:", round(metrics_21["pr_auc_mean"], 3), "| cut-offs:", round(CUT_DECLINE_21, 3), round(CUT_REVIEW_21, 3))

# 2. Fit on dev, score the holdout once
model_21 = make_model("logreg", *final_cols).fit(X_sel_filled, y_dev)
p_hold_21 = pd.Series(model_21.predict_proba(X_hold)[:, 1], index=X_hold.index)
print("HOLDOUT (21 features)  PR-AUC:", round(average_precision_score(y_hold, p_hold_21), 3),
      "| ROC-AUC:", round(roc_auc_score(y_hold, p_hold_21), 3))

# 3. Tiers on the holdout
tier_21 = pd.Series(np.select([p_hold_21 >= CUT_DECLINE_21, p_hold_21 >= CUT_REVIEW_21], ["1 decline", "2 review"],
                              default="3 approve"), index=p_hold_21.index, name="tier")
outcome_21 = pd.crosstab(tier_21, y_hold.rename("delinquent"), margins=True)
outcome_21["delinquency_rate"] = (outcome_21[1] / outcome_21["All"]).round(3)
outcome_21